**How Python lists store data in memory:**

```
Memory:
[ptr→1] [ptr→2] [ptr→3] [ptr→4] [ptr→5]
  ↓        ↓       ↓       ↓       ↓
 (int)   (int)  (int)   (int)  (int)
```

Each element is a **pointer to a Python object**. Python objects carry overhead (type info, reference count, etc.). A list of 1 million integers uses ~8 MB just for pointers, plus ~28 bytes per integer object = ~35 MB total.

**Limitations for deep learning:**

- Extremely slow for mathematical operations (loop overhead)
- Cannot run on GPU
- No vectorized math (must loop)
- High memory overhead

In [2]:
my_list=[1,2,3,4,5]
nested_list=[[1,2], [3,4]]  # 2d list

my_list
nested_list[0][1]

2

In [6]:
# operations
print(my_list[0]);
print(my_list + [6])
new_list=[x*2 for x in my_list if x%2==0]
new_list

1
[1, 2, 3, 4, 5, 6]


[4, 8]

### NumPy Arrays

NumPy is a library for **numerical computing in Python**. Its `ndarray` (N-dimensional array) stores data in a **contiguous block of memory** of a single type.

**How NumPy stores data:**

```
Memory (contiguous):
[1.0][2.0][3.0][4.0][5.0]
 f32  f32  f32  f32  f32
```

All elements are the same type, packed tightly. Operations can use SIMD (CPU vectorization) and are implemented in C — very fast.

**Limitations for deep learning:**

- Still CPU-only (no GPU support)
- No automatic gradient computation
- Not designed for deep learning workflows

In [15]:
import numpy as np
# Creating arrays
arr=np.array([1,2,3,4,5])
# matrix
matrix=np.array([[1,2], [3,4]])
# shape and dimensions
print(arr.shape)
print(matrix.shape)
print(matrix.ndim)
print(matrix.dtype)

# vectorise operations
arr2=arr *2
arr3=np.sqrt(arr)
dot=matrix @ matrix
print(arr2)
print(dot)
print(arr3)

(5,)
(2, 2)
2
int64
[ 2  4  6  8 10]
[[ 7 10]
 [15 22]]
[1.         1.41421356 1.73205081 2.         2.23606798]


### PyTorch Tensors

A PyTorch tensor is like a NumPy array but with two superpowers:

1. **Can live on GPU** — operations run in parallel on thousands of GPU cores
2. **Tracks operations for automatic differentiation** (autograd)

In [7]:
import torch
# creating tensors
t1=torch.tensor([1,2,3,4,5], dtype=torch.float32)
t2=torch.zeros(3, 4)
t3=torch.ones(2, 3, 4)
t4=torch.randn(5,5)
t5=torch.arange(0, 10, step=2)
print(t1)
print(t2)
print(t3)
print(t4)
print(t5)

tensor([1., 2., 3., 4., 5.])
tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
tensor([[[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]],

        [[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]]])
tensor([[-1.3100, -0.2381,  1.1920, -0.9351, -0.9076],
        [-0.8989,  0.0600,  0.7040, -0.4642, -0.6538],
        [ 0.9445, -1.1266, -0.5046,  0.2186, -0.3787],
        [-1.0148, -0.7605,  1.5248,  1.0168, -0.2400],
        [-0.2947, -1.3716,  2.0656, -0.9668,  0.3112]])
tensor([0, 2, 4, 6, 8])


In [8]:
print(t4.shape)
print(t4.dtype)
print(t4.device)

torch.Size([5, 5])
torch.float32
cpu


## The Critical Comparison

In [11]:
import numpy as np
import torch
import time

# Performance comparison: adding two large arrays

size=10_000_000
print(size)

10000000


In [13]:
# Python list
list_a=list(range(size))
list_b=list(range(size))
start=time.time()
list_c=[a+b for a, b in zip(list_a, list_b)]
print(f"Python list: {time.time() - start:.3f}s")

Python list: 1.199s


In [16]:
# Numpy array
np_a=np.arange(size, dtype=np.float32)
np_b=np.arange(size, dtype=np.float32)
start=time.time()
np_c=np_a+np_b
print(f"Numpy: {time.time() - start:.3f}s")

Numpy: 0.028s


In [19]:
# Pytorch CPU
t_a=torch.arange(size, dtype=torch.float32)
t_b=torch.arange(size, dtype=torch.float32)
start=time.time()
t_c=t_a+t_b
print(f"Pytorch CPU:  {time.time() - start:.3f}s")

Pytorch CPU:  0.012s


In [22]:
#Pytorch GPU (IF AVAILABLE)
if torch.cuda.is_available():
    t_a_gpu=t_a.cuda()
    t_b_gpu=t_b.cuda()
    torch.cuda.synchronize()
    start=time.time()
    t_c_gpu=t_a_gpu + t_b_gpu
    torch.cuda.synchronise()
    print(f"Pytorch GPU: {time.time - start:.3f}s")
print("No cuda available")

No cuda available


## Converting between them


In [26]:
import numpy as np
import torch

# pythoon list ->Tensor
lst=[1.0, 2.0, 3.0]
t=torch.tensor(lst)

# Numpy -> Tensor
arr=np.array([1.0, 2.0, 3.0])
t=torch.from_numpy(arr) # shares memory (no copy)
print(arr)
print(t)

[1. 2. 3.]
tensor([1., 2., 3.], dtype=torch.float64)


In [28]:
t=torch.tensor(arr)

#  Tensor -> Numpy
arr=t.numpy()  # Only works for CPU tensors
# tensor ->python list
lst=t.tolist()
print(arr)

[1. 2. 3.]


In [32]:
# important : from_numpy shares memory
arr=np.array([1.0, 2.0, 3.0])
t=torch.from_numpy(arr)
arr[0]=99.0
print(t) # tensor changed too

tensor([99.,  2.,  3.], dtype=torch.float64)


# tensor attributes you must know

```python
t = torch.randn(3, 4, requires_grad=True)

t.shape         # Shape: torch.Size([3, 4])
t.size()        # Same as shape
t.dtype         # Data type: torch.float32
t.device        # Where it lives: device(type='cpu')
t.requires_grad # Whether gradients are tracked: True
t.grad          # Stored gradient (None until .backward() called)
t.is_cuda       # Whether on GPU: False
t.ndim          # Number of dimensions: 2
t.numel()       # Total number of elements: 12
t.T             # Transpose

```

## 6. Tensor Operations for Deep Learning

This section covers the operations you'll use every single day when building deep learning models. Each one
is explained with why it matters for neural networks.

## 6.1 Creating Tensors

In [40]:
import torch
# From data
t=torch.tensor([[1, 2], [3,4]], dtype=torch.float32)
# common initilizatios
zeros=torch.zeros(3,4)
ones=torch.ones(3,4)
eye=torch.eye(4)
rand=torch.rand(3,4)
randn=torch.randn(3, 4)
empty=torch.empty(3,4)
full=torch.full((300,400), 7.0)

# print the results and see
print(zeros)
print(ones)
print(eye)
print(rand)
print(randn)

tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])
tensor([[1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.]])
tensor([[0.1206, 0.9641, 0.4223, 0.5420],
        [0.4083, 0.3536, 0.1294, 0.0342],
        [0.0595, 0.0453, 0.9086, 0.9778]])
tensor([[-0.1837,  0.1457, -0.8018,  0.1582],
        [-0.8165,  0.4319, -1.0080, -0.9134],
        [ 0.7907,  0.8513,  1.0937,  1.7904]])


In [38]:
print(empty)

tensor([[1.4013e-45, 0.0000e+00, 1.4013e-45, 0.0000e+00],
        [1.4013e-45, 0.0000e+00, 1.4013e-45, 0.0000e+00],
        [1.4013e-45, 0.0000e+00, 0.0000e+00, 0.0000e+00]])


In [41]:
print(full)

tensor([[7., 7., 7.,  ..., 7., 7., 7.],
        [7., 7., 7.,  ..., 7., 7., 7.],
        [7., 7., 7.,  ..., 7., 7., 7.],
        ...,
        [7., 7., 7.,  ..., 7., 7., 7.],
        [7., 7., 7.,  ..., 7., 7., 7.],
        [7., 7., 7.,  ..., 7., 7., 7.]])
